In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
LANDING_SCHEMA = "00_landing"
BRONZE_SCHEMA = "01_bronze"

LANDING_BASE = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/landing"
CHECKPOINT_BASE = f"{LANDING_BASE}/checkpoints/bronze_autoloader"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

sources = [
    {
        "name": "customers",
        "source_path": f"{LANDING_BASE}/customers",
        "checkpoint_path": f"{CHECKPOINT_BASE}/customers",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_customers"
    },
    {
        "name": "geolocation",
        "source_path": f"{LANDING_BASE}/geolocation",
        "checkpoint_path": f"{CHECKPOINT_BASE}/geolocation",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_geolocation"
    },
    {
        "name": "order_items",
        "source_path": f"{LANDING_BASE}/order_items",
        "checkpoint_path": f"{CHECKPOINT_BASE}/order_items",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_order_items"
    },
    {
        "name": "payments",
        "source_path": f"{LANDING_BASE}/payments",
        "checkpoint_path": f"{CHECKPOINT_BASE}/payments",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments"
    },
    {
        "name": "reviews",
        "source_path": f"{LANDING_BASE}/reviews",
        "checkpoint_path": f"{CHECKPOINT_BASE}/reviews",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_reviews"
    },
    {
        "name": "orders",
        "source_path": f"{LANDING_BASE}/orders",
        "checkpoint_path": f"{CHECKPOINT_BASE}/orders",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders"
    },
    {
        "name": "products",
        "source_path": f"{LANDING_BASE}/products",
        "checkpoint_path": f"{CHECKPOINT_BASE}/products",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_products"
    },
    {
        "name": "sellers",
        "source_path": f"{LANDING_BASE}/sellers",
        "checkpoint_path": f"{CHECKPOINT_BASE}/sellers",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sellers"
    },
    {
        "name": "category_translation",
        "source_path": f"{LANDING_BASE}/category_translation",
        "checkpoint_path": f"{CHECKPOINT_BASE}/category_translation",
        "table_name": f"{CATALOG}.{BRONZE_SCHEMA}.bronze_category_translation"
    }
]

queries = []

for src in sources:
    print(f"Starting Auto Loader for {src['name']}")
    print(f"Source: {src['source_path']}")
    print(f"Target: {src['table_name']}")
    print(f"Checkpoint: {src['checkpoint_path']}")

    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", src["checkpoint_path"] + "/schema")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(src["source_path"])
    )

    # Remove _rescued_data and source_folder columns if present
    expected_cols = [col for col in stream_df.columns if col not in ["_rescued_data", "source_folder"]]
    bronze_df = (
        stream_df.select(*expected_cols)
        .withColumn("ingestion_time", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
    )

    query = (
        bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", src["checkpoint_path"] + "/stream")
        .trigger(availableNow=True)
        .toTable(src["table_name"])
    )

    queries.append(query)

for q in queries:
    q.awaitTermination()

print("All Bronze Auto Loader streams completed.")